# cALorie — Colab landmark extraction (MediaPipe Tasks)

Clean Colab workflow: KaggleHub downloads the public workout dataset into temporary Colab storage, the current MediaPipe **Pose Landmarker Tasks API** extracts poses, and a small ZIP is saved to Google Drive.

The first run processes 2 videos from each of 7 classes (14 videos total).


In [ ]:
# Install packages in this Colab runtime.
%pip install -q --upgrade kagglehub mediapipe==0.10.35


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import os
import shutil
import urllib.request

import cv2
import kagglehub
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

print("MediaPipe:", mp.__version__)
print("Temporary disk:")
os.system("df -h /content | tail -1")


In [ ]:
# KaggleHub reuses its cache when the dataset already exists in this runtime.
DATASET_ROOT = Path(kagglehub.dataset_download(
    "hasyimabdillah/workoutfitness-video"
))
OUTPUT_ROOT = Path("/content/calorie_landmarks_smoke")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Safe first test. Change to None only after this test succeeds.
MAX_VIDEOS_PER_CLASS = 2
TARGET_FPS = 15

CLASS_MAP = {
    "barbell biceps curl": "bicep_curl",
    "lateral raise": "lateral_raise",
    "plank": "plank",
    "pull Up": "pull_up",
    "push-up": "push_up",
    "shoulder press": "shoulder_press",
    "squat": "squat",
}
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".webm"}

print("Dataset path:", DATASET_ROOT)
print("Dataset ready:", DATASET_ROOT.exists())
assert DATASET_ROOT.exists(), "Dataset download/path failed."


In [ ]:
# Download Google's official current Full pose model.
MODEL_PATH = Path("/content/pose_landmarker_full.task")
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_full/float16/latest/pose_landmarker_full.task"
)
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Pose model ready:", MODEL_PATH.exists(), MODEL_PATH)


In [ ]:
selected = []
for source_name, canonical_name in CLASS_MAP.items():
    class_dir = DATASET_ROOT / source_name
    class_videos = sorted(
        p for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    )
    if MAX_VIDEOS_PER_CLASS is not None:
        class_videos = class_videos[:MAX_VIDEOS_PER_CLASS]
    selected.extend((p, source_name, canonical_name) for p in class_videos)
    print(f"{source_name:25s} -> {canonical_name:18s}: {len(class_videos)}")

print("Total selected videos:", len(selected))
assert len(selected) == 14, "Smoke test should select 14 videos."


MediaPipe returns 33 joints. We retain the 17 joints that can be aligned with MM-Fit, then center every frame at the hips and scale it by torso size.


In [ ]:
KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    shoulder_width = np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])
    torso_length = np.linalg.norm(shoulder_center - hip_center)
    scale = max(float(shoulder_width), float(torso_length), 1e-6)
    xyz = (xyz - hip_center) / scale
    return np.concatenate([xyz, frame[:, 3:4]], axis=1)


def fill_missing(sequence):
    shape = sequence.shape
    flat = sequence.reshape(shape[0], -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction="both").fillna(0).to_numpy()
    return filled.reshape(shape).astype(np.float32)


def file_fingerprint(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    size = path.stat().st_size
    h.update(str(size).encode())
    with path.open("rb") as f:
        h.update(f.read(block_size))
        if size > block_size:
            f.seek(max(0, size - block_size))
            h.update(f.read(block_size))
    return h.hexdigest()


def extract_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, {"status": "open_failed"}

    source_fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    source_fps = source_fps if source_fps > 0 else 30.0
    sample_interval = 1.0 / TARGET_FPS
    next_sample_s = 0.0
    frame_index = 0
    poses, timestamps, detected = [], [], 0

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )

    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            time_s = frame_index / source_fps
            frame_index += 1
            if time_s + 1e-9 < next_sample_s:
                continue
            next_sample_s += sample_interval
            timestamp_ms = int(round(time_s * 1000))

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]
                pose17 = np.asarray([
                    [landmarks[i].x, landmarks[i].y, landmarks[i].z,
                     float(landmarks[i].visibility or 0.0)]
                    for i in KEEP_17
                ], dtype=np.float32)
                poses.append(normalize_pose(pose17))
                detected += 1
            else:
                poses.append(np.full((17, 4), np.nan, dtype=np.float32))
            timestamps.append(time_s)

    cap.release()
    if not poses:
        return None, {"status": "no_frames"}

    sequence = fill_missing(np.stack(poses))
    detection_rate = detected / len(sequence)
    return sequence, {
        "status": "ok" if detection_rate >= 0.5 else "low_detection",
        "source_fps": source_fps,
        "sampled_frames": len(sequence),
        "detection_rate": detection_rate,
        "duration_s": timestamps[-1] if timestamps else 0.0,
        "timestamps_s": np.asarray(timestamps, dtype=np.float32),
    }


In [ ]:
manifest_rows = []
for video_path, source_class, canonical_class in tqdm(selected, desc="Extracting poses"):
    relative_path = video_path.relative_to(DATASET_ROOT)
    clip_id = hashlib.sha1(str(relative_path).encode()).hexdigest()[:16]
    landmarks, info = extract_video(video_path)
    output_path = OUTPUT_ROOT / canonical_class / f"{clip_id}.npz"

    if landmarks is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        timestamps_s = info.pop("timestamps_s")
        np.savez_compressed(
            output_path,
            landmarks=landmarks,
            timestamps_s=timestamps_s,
            label=canonical_class,
            source_dataset="workoutfitness-video",
            original_path=str(relative_path),
            target_fps=TARGET_FPS,
        )

    manifest_rows.append({
        "clip_id": clip_id,
        "source_dataset": "workoutfitness-video",
        "source_class": source_class,
        "canonical_class": canonical_class,
        "original_path": str(relative_path),
        "output_path": str(output_path.relative_to(OUTPUT_ROOT)) if landmarks is not None else "",
        "fingerprint": file_fingerprint(video_path),
        **{k: v for k, v in info.items() if k != "timestamps_s"},
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
manifest


In [ ]:
print("Status counts:")
print(manifest["status"].value_counts(dropna=False))
print()
print("Detection quality by class:")
print(manifest.groupby("canonical_class")["detection_rate"].agg(["count", "mean", "min"]).round(3))

zip_path = shutil.make_archive(
    "/content/calorie_landmarks_smoke", "zip", root_dir=OUTPUT_ROOT
)
print()
print("ZIP ready:", zip_path)
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))


## Save permanently

The raw Kaggle videos live only in the Colab runtime. The next cell copies the compact landmark ZIP to Google Drive.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
destination = Path("/content/drive/MyDrive/cALorie-data/processed")
destination.mkdir(parents=True, exist_ok=True)
saved_path = shutil.copy2(zip_path, destination / Path(zip_path).name)
print("Saved permanently:", saved_path)


After the smoke test succeeds, change `MAX_VIDEOS_PER_CLASS = 2` to `None` and use `calorie_landmarks_v1` as the output name to process all 234 relevant videos.
